# Alethe — Integrations end-to-end

Full pipeline: real Delta + Iceberg tables → watermarks → dbt manifest lineage → twice-temporal correction → OpenLineage event emission and roundtrip.

**Install:** `pip install -e ".[all]"` from the repo root.

Sections:
1. Compute watermarks from real tables (Delta + Iceberg)
2. dbt manifest — parse a synthetic DAG, compose watermarks, run the PIT report
3. Twice-temporal correction — wire in `run_results.json`, detect non-conformance
4. OpenLineage — emit a run event, parse it back, verify the roundtrip

In [1]:
import alethe
print(f"alethe {alethe.__version__}")

alethe 0.1.0


---
## 1. Compute watermarks from real tables

Build a Delta `orders` table and an Iceberg `customers` table, vacuum / expire
their history, then run the oracle on each.

In [2]:
from pathlib import Path
import shutil, pandas as pd, pyarrow as pa
from deltalake import DeltaTable, write_deltalake

DELTA_TABLE = Path(".") / "lakehouse" / "int_orders"
DELTA_TABLE.parent.mkdir(parents=True, exist_ok=True)

for v in range(6):
    write_deltalake(
        DELTA_TABLE,
        pd.DataFrame({"order_id": [f"O{v}{i}" for i in range(4)],
                      "customer_id": [f"C{i%3}" for i in range(4)],
                      "amount": [100.0*(v+1)+i for i in range(4)]}),
        mode="overwrite",
    )

removed = DeltaTable(str(DELTA_TABLE)).vacuum(
    retention_hours=0, enforce_retention_duration=False, dry_run=False)
print(f"Delta: 6 versions written, {len(removed)} files vacuumed → {DELTA_TABLE}")

Delta: 6 versions written, 5 files vacuumed → lakehouse/int_orders


In [3]:
from pyiceberg.catalog.sql import SqlCatalog

ICE_WH = Path(".") / "iceberg_warehouse_int"
if ICE_WH.exists():
    shutil.rmtree(ICE_WH)
ICE_WH.mkdir()

catalog = SqlCatalog("local",
                     uri=f"sqlite:///{ICE_WH}/catalog.db",
                     warehouse=f"file://{ICE_WH.resolve()}")
catalog.create_namespace("raw")
tbl = catalog.create_table(
    "raw.customers",
    pa.schema([("customer_id", pa.string()), ("segment", pa.string())])
)
for s in range(5):
    tbl.overwrite(pa.table({"customer_id": [f"C{i}" for i in range(3)],
                            "segment": ["smb", "enterprise", "smb"]}))

# Destroy old snapshots out-of-band
snaps = sorted(tbl.metadata.snapshots, key=lambda s: s.sequence_number)
keep = {t.file.file_path
        for snap in snaps[-2:]
        for t in tbl.scan(snapshot_id=snap.snapshot_id).plan_files()}
removed_ice = 0
for snap in snaps[:-2]:
    try:
        for t in tbl.scan(snapshot_id=snap.snapshot_id).plan_files():
            p = Path(t.file.file_path.replace("file://", ""))
            if t.file.file_path not in keep and p.exists():
                p.unlink(); removed_ice += 1
    except Exception:
        pass
print(f"Iceberg: 5 snapshots written, {removed_ice} files deleted → raw.customers")

Iceberg: 5 snapshots written, 4 files deleted → raw.customers


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyiceberg/table/__init__.py:639: UserWarning: Delete operation did not match any records
  self.delete(


In [4]:
wm_orders    = alethe.watermark(DELTA_TABLE)
wm_customers = alethe.watermark("raw.customers", adapter="iceberg", catalog=catalog)

for wm in [wm_orders, wm_customers]:
    print(f"{wm.chain}")
    print(f"  boundary:   {wm.boundary_dt.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    print(f"  earliest:   {wm.earliest_dt.strftime('%Y-%m-%d %H:%M:%S %Z')}")
    print(f"  grade:      {wm.evidence_grade}")
    print(f"  validated:  {wm.empirically_validated}")

delta://int_orders
  boundary:   2026-07-04 15:24:01 UTC
  earliest:   2026-07-04 15:24:01 UTC
  grade:      EvidenceGrade.DERIVED
  validated:  True
iceberg://raw.customers
  boundary:   2026-07-04 15:24:01 UTC
  earliest:   2026-07-04 15:24:01 UTC
  grade:      EvidenceGrade.DERIVED
  validated:  True


---
## 2. dbt manifest — parse DAG and compose watermarks

A real dbt project produces `target/manifest.json` after `dbt compile` or `dbt run`.
Here we write a synthetic one that mirrors a typical three-layer project:

```
source: raw.orders        source: raw.customers    snapshot: snap_customers
        │                          │                         │
  stg_orders                 stg_customers ─────────────────┘
        └──────────┬──────────────┘
              revenue_summary   ← the model we're reporting on
```

`snap_customers` uses a `check` strategy — it does **not** preserve a true
PIT capture and will trigger a warning.

In [5]:
import json

MANIFEST_PATH    = Path(".") / "int_manifest.json"
RUN_RESULTS_PATH = Path(".") / "int_run_results.json"

manifest = {
    "metadata": {"dbt_schema_version": "https://schemas.getdbt.com/dbt/manifest/v10/manifest.json"},
    "nodes": {
        "model.analytics.revenue_summary": {
            "unique_id": "model.analytics.revenue_summary",
            "name": "revenue_summary",
            "resource_type": "model",
            "depends_on": {"nodes": ["model.analytics.stg_orders",
                                     "model.analytics.stg_customers"]},
        },
        "model.analytics.stg_orders": {
            "unique_id": "model.analytics.stg_orders",
            "name": "stg_orders",
            "resource_type": "model",
            "depends_on": {"nodes": ["source.analytics.raw.orders"]},
        },
        "model.analytics.stg_customers": {
            "unique_id": "model.analytics.stg_customers",
            "name": "stg_customers",
            "resource_type": "model",
            "depends_on": {"nodes": ["source.analytics.raw.customers",
                                     "snapshot.analytics.snap_customers"]},
        },
        # Snapshot with 'check' strategy — not a true PIT capture
        "snapshot.analytics.snap_customers": {
            "unique_id": "snapshot.analytics.snap_customers",
            "name": "snap_customers",
            "resource_type": "snapshot",
            "config": {"strategy": "check", "unique_key": "customer_id"},
            "depends_on": {"nodes": ["source.analytics.raw.customers"]},
        },
    },
    "sources": {
        "source.analytics.raw.orders": {
            "unique_id": "source.analytics.raw.orders",
            "source_name": "raw",
            "name": "orders",
            "schema": "raw",
            "resource_type": "source",
        },
        "source.analytics.raw.customers": {
            "unique_id": "source.analytics.raw.customers",
            "source_name": "raw",
            "name": "customers",
            "schema": "raw",
            "resource_type": "source",
        },
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
print(f"Wrote {MANIFEST_PATH}")

Wrote int_manifest.json


In [6]:
import warnings
from alethe.integrations import DbtLineage

lineage = DbtLineage(MANIFEST_PATH)
print(lineage)
print(f"\nModels:      {lineage.models()}")
print(f"Leaf nodes:  {lineage.leaf_nodes()}")

print("\nUpstream leaves of revenue_summary:")
for node in lineage.upstream_leaves("revenue_summary"):
    print(f"  {node['unique_id']}  (type={node['resource_type']})")

DbtLineage('int_manifest.json', 3 models, 3 leaf nodes, schema='https://schemas.getdbt.com/dbt/manifest/v10/manifest.json')

Models:      ['revenue_summary', 'stg_orders', 'stg_customers']
Leaf nodes:  ['snapshot.analytics.snap_customers', 'source.analytics.raw.orders', 'source.analytics.raw.customers']

Upstream leaves of revenue_summary:
  source.analytics.raw.orders  (type=source)
  source.analytics.raw.customers  (type=source)
  snapshot.analytics.snap_customers  (type=snapshot)


In [7]:
# Map leaf node unique_ids to watermarks.
# The snapshot has no live table here, so we use wm_customers as a stand-in.
watermarks = {
    "source.analytics.raw.orders":    wm_orders,
    "source.analytics.raw.customers": wm_customers,
    "snapshot.analytics.snap_customers": wm_customers,  # stand-in
}

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    report = lineage.pit_report("revenue_summary", watermarks=watermarks)

for w in caught:
    print(f"⚠ {w.message}")

print()
print(report)

⚠ Snapshot 'snap_customers' uses strategy 'check'. Only 'timestamp' strategy preserves a true PIT capture; 'check' strategy records current-state diffs and cannot reconstruct intermediate states. Treat as BOUNDED.

PIT Achievability Report: revenue_summary
────────────────────────────────────────────────────────
Upstream chain                       Boundary                 Grade
  delta://int_orders                 2026-07-04 15:24:01      EvidenceGrade.DERIVED
  iceberg://raw.customers            2026-07-04 15:24:01      EvidenceGrade.DERIVED ← limiting
  iceberg://raw.customers            2026-07-04 15:24:01      EvidenceGrade.DERIVED ← limiting
────────────────────────────────────────────────────────
Effective boundary:  2026-07-04 15:24:01  (limiting: iceberg://raw.customers)
Effective grade:     EvidenceGrade.DERIVED

PIT zones:
  CERTAIN        2026-07-04 15:24:01 → now
  BOUNDED        2026-07-04 15:24:01 → 2026-07-04 15:24:01  limiting: ['iceberg://raw.customers', 'iceberg://ra

---
## 3. Twice-temporal correction

The report above has no `Materialization` line — we haven't told it when
`revenue_summary` was last run.  Now we supply `run_results.json`.

We'll write two scenarios:
- **Conformant**: the model was run *after* the upstream retention boundary
- **Non-conformant**: the model was run *before* the boundary (stale build)

In [8]:
from datetime import timedelta

def make_run_results(completed_at: str) -> None:
    run_results = {
        "metadata": {},
        "results": [{
            "unique_id": "model.analytics.revenue_summary",
            "status": "success",
            "timing": [
                {"name": "compile",
                 "started_at": completed_at, "completed_at": completed_at},
                {"name": "execute",
                 "started_at": completed_at, "completed_at": completed_at},
            ],
        }],
    }
    RUN_RESULTS_PATH.write_text(json.dumps(run_results, indent=2))

effective_boundary = report.effective_boundary

# --- Conformant: materialized 1 hour after the boundary ---
conformant_ts = (effective_boundary + timedelta(hours=1)).isoformat()
make_run_results(conformant_ts)

conformant_report = lineage.pit_report(
    "revenue_summary",
    watermarks=watermarks,
    run_results_path=RUN_RESULTS_PATH,
)
print("=== CONFORMANT BUILD ===")
print(conformant_report)

=== CONFORMANT BUILD ===
PIT Achievability Report: revenue_summary
────────────────────────────────────────────────────────
Upstream chain                       Boundary                 Grade
  delta://int_orders                 2026-07-04 15:24:01      EvidenceGrade.DERIVED
  iceberg://raw.customers            2026-07-04 15:24:01      EvidenceGrade.DERIVED ← limiting
  iceberg://raw.customers            2026-07-04 15:24:01      EvidenceGrade.DERIVED ← limiting
────────────────────────────────────────────────────────
Effective boundary:  2026-07-04 15:24:01  (limiting: iceberg://raw.customers)
Effective grade:     EvidenceGrade.DERIVED
Materialization:     2026-07-04 16:24:01  [twice-temporal: CONFORMANT]

PIT zones:
  CERTAIN        2026-07-04 15:24:01 → now
  BOUNDED        2026-07-04 15:24:01 → 2026-07-04 15:24:01  limiting: ['iceberg://raw.customers', 'iceberg://raw.customers']
  UNACHIEVABLE   −∞ → 2026-07-04 15:24:01  limiting: ['iceberg://raw.customers']


/var/folders/w6/gzcyzpy91z7_nctprnbr1dy00000gn/T/ipykernel_83889/470894118.py:25: UserWarning: Snapshot 'snap_customers' uses strategy 'check'. Only 'timestamp' strategy preserves a true PIT capture; 'check' strategy records current-state diffs and cannot reconstruct intermediate states. Treat as BOUNDED.
  conformant_report = lineage.pit_report(


In [9]:
# --- Non-conformant: materialized 1 hour BEFORE the boundary ---
stale_ts = (effective_boundary - timedelta(hours=1)).isoformat()
make_run_results(stale_ts)

stale_report = lineage.pit_report(
    "revenue_summary",
    watermarks=watermarks,
    run_results_path=RUN_RESULTS_PATH,
)
print("=== NON-CONFORMANT BUILD ===")
print(stale_report)
print(f"\nmaterialization_conformant = {stale_report.materialization_conformant}")

=== NON-CONFORMANT BUILD ===
PIT Achievability Report: revenue_summary
────────────────────────────────────────────────────────
Upstream chain                       Boundary                 Grade
  delta://int_orders                 2026-07-04 15:24:01      EvidenceGrade.DERIVED
  iceberg://raw.customers            2026-07-04 15:24:01      EvidenceGrade.DERIVED ← limiting
  iceberg://raw.customers            2026-07-04 15:24:01      EvidenceGrade.DERIVED ← limiting
────────────────────────────────────────────────────────
Effective boundary:  2026-07-04 15:24:01  (limiting: iceberg://raw.customers)
Effective grade:     EvidenceGrade.DERIVED
Materialization:     2026-07-04 14:24:01  [twice-temporal: NON-CONFORMANT ⚠]
  !! Model was materialized before the upstream retention boundary. Results may embed vacuumed data. (spec §6)

PIT zones:
  CERTAIN        2026-07-04 15:24:01 → now
  BOUNDED        2026-07-04 15:24:01 → 2026-07-04 15:24:01  limiting: ['iceberg://raw.customers', 'iceberg://

/var/folders/w6/gzcyzpy91z7_nctprnbr1dy00000gn/T/ipykernel_83889/1825868415.py:5: UserWarning: Snapshot 'snap_customers' uses strategy 'check'. Only 'timestamp' strategy preserves a true PIT capture; 'check' strategy records current-state diffs and cannot reconstruct intermediate states. Treat as BOUNDED.
  stale_report = lineage.pit_report(


---
## 4. OpenLineage — emit, inspect, roundtrip

`to_run_event()` builds a complete OL `RunEvent` dict.  In production you'd
POST this to Marquez or Astronomer Observe from an Airflow on-success callback
or a dbt `on-run-end` hook.  Here we inspect the event and verify the
`from_facet()` roundtrip reconstructs the original watermark.

In [10]:
from alethe.integrations import to_run_event, to_facet, from_facet

event = to_run_event(
    job_name="revenue_summary",
    job_namespace="analytics",
    inputs=[wm_orders, wm_customers],
    output_name="analytics.revenue_summary",
    output_namespace="warehouse",
)

print(json.dumps(event, indent=2, default=str))

{
  "eventType": "COMPLETE",
  "eventTime": "2026-07-04T15:24:01.441869+00:00",
  "run": {
    "runId": "5d3b6dd7-3297-4ca5-85ce-2204e1b5bc19"
  },
  "job": {
    "namespace": "analytics",
    "name": "revenue_summary"
  },
  "inputs": [
    {
      "namespace": "delta",
      "name": "int_orders",
      "facets": {
        "observabilityWatermark": {
          "_producer": "https://github.com/seamus-aran/alethe",
          "_schemaURL": "https://raw.githubusercontent.com/seamus-aran/alethe/main/alethe/integrations/openlineage.py",
          "chain": "delta://int_orders",
          "boundary": {
            "version": 5
          },
          "boundaryTime": "2026-07-04T15:24:01.105000+00:00",
          "earliestTime": "2026-07-04T15:24:01.093000+00:00",
          "evidenceGrade": "derived",
          "empiricallyValidated": true,
          "claimRecordedAt": "2026-07-04T15:24:01.416423+00:00",
          "readableIslands": [],
          "proof": {
            "derivation_file_existence

In [11]:
# In Airflow this would be:
#   import requests
#   requests.post("http://marquez:5000/api/v1/lineage", json=event)
#
# For the demo we just inspect the facet and verify the roundtrip.

orders_input = next(d for d in event["inputs"] if "orders" in d["name"])
wm_orders_rt = from_facet(orders_input["name"], orders_input["facets"])

print("Roundtrip watermark:")
print(f"  chain:               {wm_orders_rt.chain}")
print(f"  boundary_dt:         {wm_orders_rt.boundary_dt}")
print(f"  evidence_grade:      {wm_orders_rt.evidence_grade}")
print(f"  empirically_valid:   {wm_orders_rt.empirically_validated}")
print()
print(f"Roundtrip boundary matches original: "
      f"{wm_orders_rt.boundary_dt == wm_orders.boundary_dt}")

Roundtrip watermark:
  chain:               delta://int_orders
  boundary_dt:         2026-07-04 15:24:01.105000+00:00
  evidence_grade:      EvidenceGrade.DERIVED
  empirically_valid:   True

Roundtrip boundary matches original: True


In [12]:
# Full circle: reconstruct a pit_report from the OL event inputs alone,
# without re-running the oracle. This is how a downstream consumer
# (e.g. a BI tool reading from Marquez) would check query achievability.

wms_from_event = [
    from_facet(d["name"], d["facets"])
    for d in event["inputs"]
]
report_from_event = alethe.pit_report("analytics.revenue_summary", wms_from_event)
print(report_from_event)

PIT Achievability Report: analytics.revenue_summary
────────────────────────────────────────────────────────
Upstream chain                       Boundary                 Grade
  delta://int_orders                 2026-07-04 15:24:01      EvidenceGrade.DERIVED
  iceberg://raw.customers            2026-07-04 15:24:01      EvidenceGrade.DERIVED ← limiting
────────────────────────────────────────────────────────
Effective boundary:  2026-07-04 15:24:01  (limiting: iceberg://raw.customers)
Effective grade:     EvidenceGrade.DERIVED

PIT zones:
  CERTAIN        2026-07-04 15:24:01 → now
  BOUNDED        2026-07-04 15:24:01 → 2026-07-04 15:24:01  limiting: ['iceberg://raw.customers']
  UNACHIEVABLE   −∞ → 2026-07-04 15:24:01  limiting: ['iceberg://raw.customers']
